In [ ]:
from preprocessing import *
from network_building import *
from generalised_euclidean import *
from multidimensional_scaling import *
from visualisations import *


In [ ]:

dates = {
    "FV22": ("2022-12-15", "2025-02-03"), # Mette Frederiksen II
    "FV19": ("2019-06-27", "2022-12-14"), # Mette Frederiksen I
    "FV15": ("2015-06-28", "2019-06-26"), # Lars Løkke Rasmussen II & III
    "FV11": ("2011-11-03", "2015-06-27") # Helle Thorning-Schmidt I & II
        }

party_colours = {
    "Alternativet": "#00FF00",
    "Danmarksdemokraterne ‒ Inger Støjberg": "#003d7f",
    "Det Konservative Folkeparti": "#00571F",
    "Kristendemokraterne": "#53619B",
    "Dansk Folkeparti": "#FCD03B",
    "Liberal Alliance": "#3FB2BE",
    "Enhedslisten – De Rød-Grønne": "#F7660D",
    "Socialdemokratiet": "#C82518",
    "Radikale Venstre": "#733280",
    "SF - Socialistisk Folkeparti": "#eb94d1",
    "Venstre, Danmarks Liberale Parti": "#01438E",
    "Nye Borgerlige": "#00505B",
    "Moderaterne": "#B48CD2",
    "Frie Grønne": "#f2e4d1",
    "Uden for partierne": "#6d6c7e",
    "Uden for folketingsgrupperne": "#6d6c7e",
    "Fremskridtspartiet": "#ffff00",
    "Borgernes Parti ‒ Lars Boje Mathiesen": "#d3c6ad",
                 }

In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
sns.set_style("whitegrid")


from tqdm import tqdm

In [ ]:
all_results = {}
for year in dates:

    print(f"\nNow handling {year}")
    # Generate baseline dataframes
    pol, cand, votes, map = pipeline_raw_data_to_dataframes(year, dates)

    if year == "FV11": # edge case
        cand = cand.loc[~((cand["aktørid"] == 202) & (cand["Candidates.Party"] == "Venstre, Danmarks Liberale Parti"))] 

    if year == "FV15":
        remove_ids = [132, 144] # edge case

        pol = pol[~pol["aktørid"].isin(remove_ids)]
        cand = cand[~cand["aktørid"].isin(remove_ids)]
        votes = votes[~votes["aktørid"].isin(remove_ids)]
        map = {k: v for k, v in map.items() if k not in remove_ids}

    # Get opinion vectors
    o_vector_votes, o_vector_cand, fit_metrics = get_coordinates_and_fit_metrics(votes, cand)

    # Generate network for both cand and votes
    print(f"\nNetwork for candidates {year}")

    network_cand = nx.from_pandas_edgelist(
        pipeline_dataframe_to_network(
            cand, "candidates", map), 
        "src", "trg", "nij")
    
    print(f"\nNetwork for votes {year}")
    network_votes = nx.from_pandas_edgelist(
        pipeline_dataframe_to_network(
            votes, "votes", map), 
        "src", "trg", "nij")
    
    
    # Calculate GE 
    print(f"\nResults for candidates {year}")
    #ugly_results_cand = normalize_ge_results(contextualise_ge_result(o_vector_votes, network_cand, map))
    results_cand = normalize_ge_results(contextualise_ge_result(o_vector_cand, network_cand, map))

    print(f"\nResults for votes {year}")
    results_votes = normalize_ge_results(contextualise_ge_result(o_vector_votes, network_votes, map))
    #ugly_results_votes = normalize_ge_results(contextualise_ge_result(o_vector_cand, network_votes, map))

    # Store results in a dictionary with descriptive keys
    all_results[f"results_cand_{year}"] = results_cand
    #all_results[f"ugly_cand_{year}"] = ugly_results_cand
    all_results[f"results_votes_{year}"] = results_votes
    #all_results[f"ugly_votes_{year}"] = ugly_results_votes


In [ ]:
all_results_df = pd.DataFrame(all_results).T.drop(columns=["polarised", "party mean"])
plot_normalised_outputs(all_results_df, (8, 5))

In [ ]:

hairball = plot_network_with_party_colors(
    nx.from_pandas_edgelist(
        calculate_agreement_pair_counts(map, votes, "afstemningid", "typeid"), 
        source="src", target="trg", edge_attr="nij"), 
    map, color_mapping=party_colours, node_size=400, figsize=(10,8)
    )

In [ ]:
party_order = [
    "Det Konservative Folkeparti",
    "Venstre, Danmarks Liberale Parti",
    "Danmarksdemokraterne ‒ Inger Støjberg",
    "Dansk Folkeparti",
    "Nye Borgerlige",
    "Liberal Alliance",
    "Moderaterne",
    "Radikale Venstre",
    "Alternativet",
    "SF - Socialistisk Folkeparti",
    "Enhedslisten – De Rød-Grønne",
    "Socialdemokratiet"
]
abbreviated_party_names = {
    "Det Konservative Folkeparti": "Konservative",
    "Venstre, Danmarks Liberale Parti": "Venstre",
    "Danmarksdemokraterne ‒ Inger Støjberg": "Danmarksdemokraterne",
    "SF - Socialistisk Folkeparti": "SF",
    "Enhedslisten – De Rød-Grønne": "Enhedslisten"
}

In [ ]:
plot_network_with_party_colors(network_votes, map, party_colours, party_order, group_label_map=abbreviated_party_names)

In [ ]:
# blocs = {
#     #"Alternativet": "Green",
#     #"Enhedslisten – De Rød-Grønne": "Green",

#     "Danmarksdemokraterne ‒ Inger Støjberg": "Blue",
#     "Det Konservative Folkeparti": "Blue",
#     "Dansk Folkeparti": "Blue",
#     "Liberal Alliance": "Blue",
#     "Venstre, Danmarks Liberale Parti": "Blue",
#     "Nye Borgerlige": "Blue",
#     "Borgernes Parti ‒ Lars Boje Mathiesen": "Blue",

#     "Alternativet": "Red",
#     "Enhedslisten – De Rød-Grønne": "Red",
#     "Socialdemokratiet": "Red",
#     "Radikale Venstre": "Red",
#     "SF - Socialistisk Folkeparti": "Red"
# }

# government_parties = {
#     "FV22": ("Socialdemokratiet", "Venstre, Danmarks Liberale Parti", "Moderaterne"), # Mette Frederiksen II
#     "FV19": ("Socialdemokratiet"), # Mette Frederiksen I
#     "FV15": ("Venstre, Danmarks Liberale Parti", "Liberal Alliance", "Det Konservative Folkeparti"), # Lars Løkke Rasmussen III
#     "FV11": ("Socialdemokratiet", "Radikale Venstre",  "SF - Socialistisk Folkeparti") # Helle Thorning-Schmidt II
# }